# High-frequency EMIC waves observed by Arase during the May 2024 Gannon storm

During the May 2024 geomagnetic superstorm, the Arase (ERG) spacecraft observed electromagnetic ion cyclotron (EMIC) waves deep in the inner magnetosphere. EMIC waves are low-frequency electromagnetic waves whose spectral bands are organized by ion gyrofrequencies. Because gyrofrequency increases with magnetic-field strength, waves observed close to Earth can occur at much higher frequencies than the more typical 0.2–5 Hz EMIC range.

In this notebook we will:

1. load Arase MGF, PWE/HFA, and orbit data with **PySPEDAS 2.2.0**;
2. inspect and uniformly resample the 256 Hz magnetic waveform;
3. use PySPEDAS `avg_data` and `subtract` to separate an ambient field from wave fluctuations;
4. construct a magnetic power spectral density (PSD) spectrogram with `tdpwrspc`;
5. calculate local H$^+$, He$^+$, and O$^+$ gyrofrequencies; and
6. combine the spectrum and gyrofrequency tplot variables in an overplot pseudovariable.

The scientific motivation is [Jun et al. (2025), *Arase In Situ Observations of High-Frequency Electromagnetic Ion Cyclotron (EMIC) Waves in Regions Close to the Earth During the May 2024 Storm*](https://doi.org/10.1029/2024GL112489), especially their Figures 1, 3, and 4. We calculate our own introductory spectrum and **local** gyrofrequencies; we do not reproduce the paper's mean-field-aligned polarization analysis or model-mapped equatorial quantities.

> **Date note:** the Figure 3 caption in the paper says May 10–11, but the surrounding text, Figure 1, and the data identify this event as **May 11–12, 2024**. We use the latter dates.


## 1. Imports and time intervals

PySPEDAS 2.2.0 exposes the data loaders, processing routines, and tplot functions used below. NumPy is retained for array inspection and a few transparent calculations; Matplotlib is used only for the one-minute frequency profile, which is not a time-series plot.


In [1]:
# This line installs PySPEDAS so the notebook can be used in Google Colab.  You can comment
# it out or skip this cell if you already have a suitable PySPEDAS installation.

!pip install "pyspedas>=2.2.0"

In [2]:
%matplotlib inline

from datetime import datetime, timezone
from pprint import pprint

import pyspedas
from pyspedas import *
import matplotlib.pyplot as plt
import numpy as np
from scipy.constants import elementary_charge, proton_mass

pyspedas.version()


02-Oct-26 18:01:54: pyspedas version: 2.2.0


In [3]:
trange = ["2024-05-11/22:40", "2024-05-12/00:10"]
wave_trange = ["2024-05-11/23:55", "2024-05-12/00:06"]
paper_psd_trange = ["2024-05-11/23:56", "2024-05-11/23:57"]

first_event = ["2024-05-11/22:47", "2024-05-11/22:57"]
second_event = wave_trange
trange, wave_trange

(['2024-05-11/22:40', '2024-05-12/00:10'],
 ['2024-05-11/23:55', '2024-05-12/00:06'])

## 2. Load a low-cadence overview

We start with small context products rather than the waveform files:

- MGF Level 2, 8 s magnetic field;
- PWE/HFA Level 3, 1 min electron density inferred from the upper-hybrid resonance; and
- Level 2 orbit data.

The loaders return the names of the tplot variables they create. Loading both dates is necessary because the interval crosses midnight.

In [4]:
erg = pyspedas.projects.erg

mgf_overview_vars = erg.mgf(
    trange=trange, datatype="8sec", level="l2", time_clip=True
)
hfa_vars = erg.pwe_hfa(
    trange=trange, level="l3", time_clip=True
)
orbit_vars = erg.orb(
    trange=trange, level="l2", datatype="def", time_clip=True
)

print("MGF variables:", mgf_overview_vars)
print("HFA variables:", hfa_vars)
print("Orbit variables:", orbit_vars)

02-Oct-26 18:01:54: Downloading remote index: https://ergsc.isee.nagoya-u.ac.jp/data/ergsc/satellite/erg/mgf/l2/8sec/2024/05/
02-Oct-26 18:02:19: Retrying (LoggingRetry(total=2, connect=None, read=None, redirect=None, status=None)) after connection broken by 'ReadTimeoutError("HTTPSConnectionPool(host='ergsc.isee.nagoya-u.ac.jp', port=443): Read timed out. (read timeout=20)")': /data/ergsc/satellite/erg/mgf/l2/8sec/2024/05/


KeyboardInterrupt: 

In [ ]:
bmag_overview = "erg_mgf_l2_magt_8sec"
density_var = "erg_pwe_hfa_l3_1min_ne_mgf"
orbit_l_var = "erg_orb_l2_pos_Lm"
orbit_position_var = "erg_orb_l2_pos_rmlatmlt"

expected_context = [bmag_overview, density_var, orbit_l_var, orbit_position_var]
returned_context = mgf_overview_vars + hfa_vars + orbit_vars
for name in expected_context:
    if name not in returned_context or get_data(name) is None:
        raise RuntimeError(f"Expected context variable {name!r} was not loaded")

# The three delivered McIlwain-L estimates differ only by pitch angle. Their
# median gives one uncluttered context trace without relying on component order.
l_data = get_data(orbit_l_var)
store_data(
    "erg_orb_L_median",
    data={"x": l_data.times, "y": np.nanmedian(l_data.y, axis=1)},
)

options(bmag_overview, "ytitle", "Arase |B|")
options(bmag_overview, "ysubtitle", "[nT], 8 s")
options(density_var, "ytitle", "Electron density")
options(density_var, "ysubtitle", "[cm^-3], 1 min")
options(density_var, "ylog", True)
options("erg_orb_L_median", "ytitle", "McIlwain L")
options("erg_orb_L_median", "ysubtitle", "IGRF")

for event_time, color in [
    (first_event[0], "gray"), (first_event[1], "gray"),
    (second_event[0], "red"), (second_event[1], "red"),
]:
    timebar(time_double(event_time), color=color, thick=1)

tplot([bmag_overview, density_var, "erg_orb_L_median"], xsize=12, ysize=8)

The gray guides mark the first reported HF EMIC interval (22:47–22:57 UT); the red guides mark the multiband event used below (23:55–00:06 UT). Before proceeding, notice the several-thousand-nT field, high density, and low L shell. These are very different conditions from a typical outer-magnetosphere EMIC event.

### Inspect context metadata

The HFA product name is not obvious from the loader call: in PySPEDAS 2.2.0 the density is `erg_pwe_hfa_l3_1min_ne_mgf`. The metadata confirm that its units are particles per cubic centimeter (`/cc`).

In [ ]:
for name in [bmag_overview, density_var, orbit_l_var, orbit_position_var]:
    metadata = get_data(name, metadata=True)
    data = get_data(name)
    cadence = np.nanmedian(np.diff(data.times))
    print(f"\n{name}: {len(data.times)} samples, median cadence {cadence:.3f} s")
    pprint({
        "units": metadata.get("data_att", {}).get("units"),
        "coordinate_system": metadata.get("data_att", {}).get("coord_sys"),
        "field_name": metadata.get("CDF", {}).get("VATT", {}).get("FIELDNAM"),
        "description": metadata.get("CDF", {}).get("VATT", {}).get("CATDESC"),
    })

## 3. Load and inspect the 256 Hz waveform

The 256 Hz MGF data are a special high-rate mode available at low L shell. Unlike the 8 s context product, these data resolve fluctuations up to a Nyquist frequency near 128 Hz. The loader uses hourly files, so this 11-minute interval downloads the 23 UT and 00 UT files on a first run; later runs use the cache.

In [ ]:
wave_vars = erg.mgf(
    trange=wave_trange,
    datatype="256hz",
    level="l2",
    coord="dsi",
    time_clip=True,
)
print("256 Hz MGF variables:", wave_vars)

wave_var = "erg_mgf_l2_mag_256hz_dsi"
if wave_var not in wave_vars or get_data(wave_var) is None:
    raise RuntimeError(f"Expected waveform variable {wave_var!r} was not loaded")

In [ ]:
wave_data = get_data(wave_var)
wave_metadata = get_data(wave_var, metadata=True)
wave_times = np.asarray(wave_data.times)
wave_b = np.asarray(wave_data.y, dtype=float)

dt = np.diff(wave_times)
# Exclude obvious gaps when estimating the delivered cadence.
short_dt = dt[dt < 0.02]
sample_dt = np.nanmedian(short_dt)
sample_rate = 1.0 / sample_dt
nyquist = sample_rate / 2.0
finite_rows = np.all(np.isfinite(wave_b), axis=1)
gap_threshold = 5.0 * sample_dt
gap_indices = np.flatnonzero(dt > gap_threshold)

print(f"Waveform shape: {wave_b.shape}")
print(f"Measured median cadence: {sample_dt * 1000:.4f} ms")
print(f"Effective sample rate: {sample_rate:.3f} Hz")
print(f"Nyquist frequency: {nyquist:.3f} Hz")
print(f"Rows with three finite components: {finite_rows.mean():.3%}")
print(f"Time gaps larger than {gap_threshold:.4f} s: {len(gap_indices)}")
for i in gap_indices:
    t0 = datetime.fromtimestamp(wave_times[i], tz=timezone.utc)
    t1 = datetime.fromtimestamp(wave_times[i + 1], tz=timezone.utc)
    print(f"  {t0.isoformat()} to {t1.isoformat()} ({dt[i]:.3f} s)")

pprint({
    "units": wave_metadata.get("data_att", {}).get("units"),
    "coordinate_system_attribute": wave_metadata.get("data_att", {}).get("coord_sys"),
    "labels": wave_metadata.get("CDF", {}).get("LABELS"),
    "field_name": wave_metadata.get("CDF", {}).get("VATT", {}).get("FIELDNAM"),
    "description": wave_metadata.get("CDF", {}).get("VATT", {}).get("CATDESC"),
})

The variable name and CDF description identify **DSI coordinates**, and the units are nT. In the current files the generic `coord_sys` metadata field is blank, which is why checking more than one metadata key matters.

The cadence is close to, but not exactly, the nominal 256 Hz when estimated from time tags. We use the measured median cadence for the frequency axis. We also keep gaps as gaps: filtering across missing data would create false ringing, and concatenating separated samples before an FFT would create spurious spectral power.

## 4. Establish the ambient field and wave fluctuations

For this event, a one-second bin average is a simple ambient-field estimate:

$$\delta\mathbf{B} = \mathbf{B} - \mathbf{B}_0, \qquad
\mathbf{B}_0 = \langle\mathbf{B}\rangle_{1\ \mathrm{s}}.$$

`avg_data(res=1.0)` makes non-overlapping one-second averages; it is not a sliding low-pass filter. One second is short enough to follow the storm-time field but averages many cycles of the 5–35 Hz emissions. In a development comparison, its $|B_0|$ differed from the previous zero-phase 1 Hz Butterworth estimate by about 0.003% typically, so the gyrofrequency tracks are essentially unchanged.

The delivered time tags have small jitter. We use `time_interpolate` to put the waveform and ambient field on an exact 256 Hz grid before subtraction and spectral analysis. `max_gap_time` prevents interpolation across the invalid block and the real gap: missing data remain missing.


In [ ]:
background_var = "arase_mgf_b0_1s_dsi"
wave_uniform_var = "arase_mgf_b_dsi_uniform"
background_uniform_var = "arase_mgf_b0_dsi_uniform"
wave_fluctuation_var = "arase_mgf_db_dsi_uniform"

# Average the three-component waveform into one-second bins.
avg_data(wave_var, res=1.0, newname=background_var)

# The product is nominally 256 Hz. An exact grid makes FFT windows uniform.
uniform_sample_rate = 256.0
grid_start = np.ceil(wave_times[0] * uniform_sample_rate) / uniform_sample_rate
grid_stop = np.floor(wave_times[-1] * uniform_sample_rate) / uniform_sample_rate
n_uniform = int(np.floor((grid_stop - grid_start) * uniform_sample_rate)) + 1
uniform_times = grid_start + np.arange(n_uniform) / uniform_sample_rate

time_interpolate(
    wave_var, uniform_times, newname=wave_uniform_var,
    method="linear", ignore_nans=True, max_gap_time=0.02,
    nan_extrapolate=True,
)
time_interpolate(
    background_var, uniform_times, newname=background_uniform_var,
    method="linear", ignore_nans=True, max_gap_time=2.0,
    nan_extrapolate=True,
)
subtract(wave_uniform_var, background_uniform_var, newname=wave_fluctuation_var)

options(wave_fluctuation_var, "legend_names", ["dBx", "dBy", "dBz"])
options(wave_fluctuation_var, "ytitle", "Wave B")
options(wave_fluctuation_var, "ysubtitle", "DSI [nT]")

uniform_wave = get_data(wave_uniform_var)
uniform_background = get_data(background_uniform_var)
wave_fluctuations = np.asarray(get_data(wave_fluctuation_var).y)
finite_uniform = np.all(np.isfinite(wave_fluctuations), axis=1)

print(f"Uniform samples: {len(uniform_times):,}")
print(f"Uniform cadence: {1000.0 / uniform_sample_rate:.6f} ms")
print(f"Finite fluctuation rows: {finite_uniform.mean():.3%}")
print("No values are interpolated across gaps longer than 0.02 s.")


In [ ]:
# A five-second tplot zoom makes the high-rate character tangible.
waveform_zoom = ["2024-05-11/23:56:25", "2024-05-11/23:56:30"]

zoom_start, zoom_stop = time_double(waveform_zoom)
fluctuation_data = get_data(wave_fluctuation_var)
zoom_mask = (
    (fluctuation_data.times >= zoom_start)
    & (fluctuation_data.times <= zoom_stop)
)

# Override the several-thousand-nT range inherited from the original MGF variable.
zoom_limit = 1.1 * np.nanpercentile(
    np.abs(fluctuation_data.y[zoom_mask]), 99.5
)
options(wave_fluctuation_var, "yrange", [-zoom_limit, zoom_limit])
options(wave_fluctuation_var, "ylog", False)

tplot(wave_fluctuation_var, trange=waveform_zoom, xsize=12, ysize=4)

## 5. Construct the magnetic PSD spectrogram with `tdpwrspc`

`tdpwrspc` applies the same analysis to every component of a vector tplot variable. We use a **4096-point Hann window stepped by 1280 points**: at 256 Hz these are 16 s and 5 s, matching the cadence reported by Jun et al. Setting `bin=1` retains the native 0.0625 Hz frequency spacing.

`notmvariance=True` rejects windows crossing a gap. It works here because `time_interpolate` first removed the harmless time-tag jitter while retaining real gaps as NaNs. We then use the PySPEDAS `add` routine to sum the three component spectra. The resulting total magnetic PSD is invariant under coordinate rotation and has units of nT$^2$/Hz.


In [ ]:
nboxpoints = 4096
nshiftpoints = 1280

component_spectra = tdpwrspc(
    wave_fluctuation_var,
    nboxpoints=nboxpoints,
    nshiftpoints=nshiftpoints,
    bin=1,
    notmvariance=True,
)
if not isinstance(component_spectra, list) or len(component_spectra) != 3:
    raise RuntimeError("Expected one tdpwrspc output for each magnetic component")

add(component_spectra[0], component_spectra[1], newname="arase_emic_psd_xy")
total_psd_var = "arase_emic_total_psd"
add("arase_emic_psd_xy", component_spectra[2], newname=total_psd_var)

psd_data = get_data(total_psd_var)
total_psd = np.asarray(psd_data.y)
frequency_maps = np.asarray(psd_data.v)
valid_spectral_rows = np.any(np.isfinite(total_psd), axis=1)
first_valid_row = np.flatnonzero(valid_spectral_rows)[0]
frequencies = (
    frequency_maps[first_valid_row]
    if frequency_maps.ndim == 2 else frequency_maps
)

# dpwrspc returns a time-dependent frequency map. Rejected gap windows have an
# all-NaN map even though every valid window uses the same uniform frequency
# grid. Restoring the common grid removes a harmless tplot bin-edge warning;
# the PSD rows themselves remain NaN and therefore remain blank.
store_data(
    total_psd_var,
    data={"x": psd_data.times, "y": total_psd, "v": frequencies},
)
psd_data = get_data(total_psd_var)

options(total_psd_var, "spec", True)
options(total_psd_var, "ytitle", "Frequency")
options(total_psd_var, "ysubtitle", "[Hz]")
options(total_psd_var, "yrange", [2, 90])
options(total_psd_var, "ylog", True)
options(total_psd_var, "zlog", True)
options(total_psd_var, "zrange", [1e-4, 2])
options(total_psd_var, "Colormap", "turbo")
options(total_psd_var, "ztitle", "PSD")
options(total_psd_var, "zsubtitle", "[nT^2/Hz]")

print(f"Window length: {nboxpoints / uniform_sample_rate:.2f} s")
print(f"Window step: {nshiftpoints / uniform_sample_rate:.2f} s")
print(f"Frequency resolution: {frequencies[1] - frequencies[0]:.4f} Hz")
print(f"Valid spectra: {valid_spectral_rows.sum()} of {len(valid_spectral_rows)}")


## 6. Calculate ion gyrofrequencies

For an ion with charge magnitude $|q_i|$ and mass $m_i$,

$$f_{ci}=\frac{|q_i|B}{2\pi m_i}.$$

MGF reports nT, so we multiply by $10^{-9}$ before using the SI formula. For singly ionized H$^+$, He$^+$, and O$^+$ the charge is the same; using mass numbers 1, 4, and 16 gives $f_{cHe}\approx f_{cH}/4$ and $f_{cO}\approx f_{cH}/16$.

These are **local** gyrofrequencies calculated from the one-second ambient-field magnitude. The paper also uses model-mapped equatorial field quantities for parts of its analysis; those are not interchangeable with the local values calculated here.


In [ ]:
background_data = get_data(background_var)
background_times = np.asarray(background_data.times)
background_b = np.asarray(background_data.y, dtype=float)
background_bmag = np.linalg.norm(background_b, axis=1)

def gyrofrequency_hz(b_nT, mass_number):
    b_tesla = np.asarray(b_nT) * 1e-9
    return elementary_charge * b_tesla / (2.0 * np.pi * mass_number * proton_mass)

gyro_vars = {
    "H+": ("arase_local_f_cH", 1, "black"),
    "He+": ("arase_local_f_cHe", 4, "orange"),
    "O+": ("arase_local_f_cO", 16, "blue"),
}

for label, (name, mass_number, color) in gyro_vars.items():
    values = gyrofrequency_hz(background_bmag, mass_number)
    store_data(name, data={"x": background_times, "y": values})
    options(name, "color", color)
    options(name, "line_width", 2)
    options(name, "legend_names", [label])
    options(name, "ytitle", "Frequency")
    options(name, "ysubtitle", "[Hz]")
    print(f"{label:3s}: {np.nanmin(values):6.2f} to {np.nanmax(values):6.2f} Hz")


## 7. Overlay the gyrofrequencies on the spectrum

The frequency axis is logarithmic so all three gyrofrequency tracks fit without hiding the 5–35 Hz structure. In a multi-ion plasma the conventional bands are:

- H$^+$ band: $f_{cHe}<f<f_{cH}$;
- He$^+$ band: $f_{cO}<f<f_{cHe}$; and
- O$^+$ band: $f<f_{cO}$.

These boundaries organize the observations, but a band location alone is not a complete wave-mode or polarization diagnosis.

In [ ]:
combined_spectrum_var = "arase_emic_psd_with_gyrofrequencies"
store_data(
    combined_spectrum_var,
    data=[total_psd_var] + [item[0] for item in gyro_vars.values()],
)
options(combined_spectrum_var, "ytitle", "Frequency")
options(combined_spectrum_var, "ysubtitle", "[Hz]")

# A pseudovariable draws all children in the same panel. The line variables
# already use the spectrogram's frequency axis, so no right-hand axis is needed.
tplot(combined_spectrum_var, trange=wave_trange, xsize=13, ysize=6.5)


The strongest packets near 5–10 Hz lie between the local O$^+$ and He$^+$ gyrofrequencies and are therefore naturally described as He$^+$-band emissions. Weaker isolated power near 20–35 Hz lies above $f_{cHe}$ and below $f_{cH}$, in the H$^+$ band. This is the unusual high-frequency part of the event emphasized by Jun et al.

Blank columns are rejected windows around the invalid interval or timing gap, not low wave power. Preserving those blanks is an important part of the analysis rather than a plotting inconvenience.


### One-minute average spectrum: 23:56–23:57 UT

Jun et al. use this minute for their Figure 4 PSD and resonance calculation. We only average our total magnetic PSD and mark the mean **local** gyrofrequencies. The published paper reports well-defined He$^+$-band power at 5–10 Hz and H$^+$-band power at 27–35 Hz.

In [ ]:
psd_start, psd_stop = time_double(paper_psd_trange)
minute_mask = (
    (np.asarray(psd_data.times) >= psd_start)
    & (np.asarray(psd_data.times) < psd_stop)
    & valid_spectral_rows
)
minute_psd = np.nanmean(total_psd[minute_mask], axis=0)

background_minute = (
    (background_times >= psd_start) & (background_times < psd_stop)
)
minute_b = np.nanmean(background_bmag[background_minute])

# A conventional frequency-profile plot is clearer than treating frequency as time.
fig, ax = plt.subplots(figsize=(10, 4.5))
frequency_mask = (frequencies >= 2) & (frequencies <= 90)
ax.semilogy(frequencies[frequency_mask], minute_psd[frequency_mask], color="black")
average_colors = {"H+": "red", "He+": "magenta", "O+": "green"}
for label, mass_number in [("H+", 1), ("He+", 4), ("O+", 16)]:
    ax.axvline(
        gyrofrequency_hz(minute_b, mass_number),
        color=average_colors[label], lw=1.5, ls="--", label=label,
    )
ax.axvspan(5, 10, color="blue", alpha=0.12, label="reported He+ band")
ax.axvspan(27, 35, color="orange", alpha=0.12, label="reported H+ band")
ax.set_xscale("log")
ax.set_xlim(2, 90)
ax.set_xticks([2, 5, 10, 20, 40, 80])
ax.get_xaxis().set_major_formatter(plt.ScalarFormatter())
ax.set_xlabel("Frequency [Hz]")
ax.set_ylabel(r"Mean PSD [nT$^2$/Hz]")
ax.set_title("Mean total magnetic PSD, 23:56–23:57 UT")
ax.grid(alpha=0.25)
ax.legend(ncol=2)
fig.tight_layout()


## 8. Density and spacecraft-location context

The HFA density is inferred from the upper-hybrid resonance rather than measured by a particle counter. The orbit product supplies geocentric radius, IGRF magnetic latitude (MLAT), magnetic local time (MLT), and McIlwain L. We show the median of the three pitch-angle-dependent L estimates.

In [ ]:
density = get_data(density_var)
position = get_data(orbit_position_var)
orbit_l = get_data(orbit_l_var)
context_start, context_stop = time_double(wave_trange)

# Position columns are geocentric radius, IGRF MLAT, and MLT.
store_data("erg_orb_mlat", data={"x": position.times, "y": position.y[:, 1]})
options("erg_orb_mlat", "ytitle", "MLAT")
options("erg_orb_mlat", "ysubtitle", "[deg]")
options("erg_orb_L_median", "ytitle", "McIlwain L")

tplot(
    [density_var, "erg_orb_L_median", "erg_orb_mlat"],
    trange=wave_trange,
    xsize=12,
    ysize=8,
)

density_mask = (density.times >= context_start) & (density.times <= context_stop)
position_mask = (position.times >= context_start) & (position.times <= context_stop)
l_mask = (orbit_l.times >= context_start) & (orbit_l.times <= context_stop)

print("Focused-interval median density [cm^-3]:", np.nanmedian(density.y[density_mask]))
print("Focused-interval density range [cm^-3]:",
      np.nanmin(density.y[density_mask]), np.nanmax(density.y[density_mask]))
print("Focused-interval median [R, MLAT, MLT]:",
      np.nanmedian(position.y[position_mask], axis=0))
print("Focused-interval median McIlwain L:",
      np.nanmedian(orbit_l.y[l_mask]))


The event occurs near L $\sim$ 2 with density of hundreds to more than one thousand cm$^{-3}$. The large local magnetic field at low L shell raises the ion gyrofrequencies, while the dense cold plasma is relevant to EMIC-wave generation and propagation. Establishing that context does not require a full dispersion-relation calculation.

The paper reports a wave-normal angle analysis and polarization properties in a mean-field-aligned coordinate system. Our summed three-component PSD is rotation invariant and is sufficient to identify the bands, but it intentionally does not classify polarization.

## 9. Exercises

1. Remove the gyrofrequency variables from `combined_spectrum_var`. At what frequencies and times do you identify the strongest isolated bands?
2. Choose one time near 23:56:30 UT. Read $|B_0|$ from `background_bmag` and calculate $f_{cH}$ by hand. Compare with `gyrofrequency_hz`.
3. Which observed packets fall naturally in the H$^+$ band and which in the He$^+$ band? Does the O$^+$ gyrofrequency pass through the strongest emission?
4. Change `nboxpoints` from 4096 to 2048 and `nshiftpoints` from 1280 to 640. How do the time and frequency resolutions change? Which structures become easier or harder to see?
5. Compare the first interval (22:47–22:57 UT) with the second. Because that requires another hourly 256 Hz file, do it as an extension rather than adding it to the default download.
6. Change the `avg_data` resolution from 1 s to 0.5 or 2 s. Are the 5–35 Hz spectrum and gyrofrequency tracks stable? Why is this a different sensitivity test from changing a Butterworth cutoff?
7. Set `notmvariance=False`, or omit `time_interpolate`, and inspect what happens near gaps or when cadence jitter is retained.
8. Do the observed bands track the changing gyrofrequency curves, or are individual packets more nearly horizontal over their short lifetimes?

**Advanced extension:** the companion polarization notebook uses `pyspedas.twavpol` on the same event after a mean-field-aligned rotation.


## 10. Scientific takeaway

Arase observed isolated magnetic-wave packets at unusually high frequencies during the May 2024 superstorm. The 256 Hz MGF waveform resolves strong 5–10 Hz power and weaker 20–35 Hz power while the spacecraft was in a dense plasma environment near L $\sim$ 2. Calculating gyrofrequencies directly from the measured ambient field organizes those emissions into He$^+$ and H$^+$ bands.

What we calculated here: a gap-aware, uniformly sampled three-component magnetic PSD using PySPEDAS; a one-second bin-averaged ambient field; local H$^+$, He$^+$, and O$^+$ gyrofrequencies; and density/orbit context. The central figure is a tplot pseudovariable combining the spectrum and gyrofrequency tracks.

What comes from Jun et al. (2025): the event selection, the interpretation as high-frequency EMIC waves, the detailed mean-field-aligned polarization and wave-normal analysis, and the reported 5–10 Hz and 27–35 Hz bands for 23:56–23:57 UT.


## Instructor notes and current caveats

- A fresh focused run downloads two hourly MGF 256 Hz files because the interval crosses midnight. In the archive version tested while developing this notebook, they were about **15.6 MB** and **11.6 MB**. The overview MGF, HFA, and orbit files add only a few MB.
- The ERG server returned all required files, but directory-index responses occasionally took tens of seconds. Reruns use the local cache.
- The current archive supplied MGF v04.06, HFA L3 v06.11, and orbit L2 v05. Jun et al. list earlier versions in their data-availability statement; small numerical or visual differences from the paper are therefore expected.
- In PySPEDAS 2.2.0, the MGF loader's `varnames` argument does not reduce the returned MGF variables. The notebook therefore validates the expected variable after loading rather than assuming a filtered return list.
- The high-rate variable is `erg_mgf_l2_mag_256hz_dsi`. Its name and CDF description identify DSI coordinates, but the generic `coord_sys` metadata entry is blank in the files tested.
- `avg_data(res=1.0)` is a non-overlapping bin average, not a centered moving average or a formal low-pass filter. It is defensible here because the scientific bands begin near 5 Hz and the several-thousand-nT ambient field changes slowly. A development comparison with the earlier 1 Hz Butterworth estimate changed $|B_0|$ by about 0.003% typically.
- Empty one-second bins at real outages remain NaN. PySPEDAS 2.2.0 may print a `Mean of empty slice` warning while constructing those bins; it does not mean the gap was filled.
- The native time tags have small cadence variations. `time_interpolate` creates a nominal 256 Hz grid, while `max_gap_time` preserves real outages as NaNs. Consequently, `tdpwrspc(..., notmvariance=True)` rejects only windows that cross gaps rather than rejecting every jittered native window.
- The 4096-point window and 1280-point step correspond to 16 s and 5 s at 256 Hz, following the cadence stated by Jun et al. The fixed tplot color limits should be rechecked if calibration versions change.
- HFA L3 density has 1 min cadence and includes a quality flag. This introductory plot shows the delivered density without interpreting the flag bit mask; research use should consult the product documentation.
- The central spectrum sums DSI-component PSDs and is invariant under rotation. It is not the paper's perpendicular/parallel mean-field-aligned spectrum and cannot by itself determine polarization or wave-normal angle.
- ERG/Arase rules of use and instrument-team citation guidance printed by the loaders should be followed for presentations and publications.
